# Prepare Quartet multiomics inputs for batch correction (step 02)

Reads the pre-filtered 72-sample expression matrices and combined metadata
written by **`01_preprocess_eda.ipynb`** from `filtered_data/`, validates the
design, and writes per-modality central and FedRBE client inputs to `before/`.

**Dependency:** `01_preprocess_eda.ipynb` must be run first to populate
`filtered_data/`.

In [1]:
library(tidyverse)
library(jsonlite)

Warning message:
“package ‘dplyr’ was built under R version 4.5.3”
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘jsonlite’


The following object is masked from ‘package:purrr’:

    flatten




In [2]:
# WORKDIR auto-detects whether we are running from the repo root or from inside
# the multiomics folder (e.g. when launched via Jupyter inside evaluation_data/).
WORKDIR      <- if (dir.exists("figshare_data")) "." else "evaluation_data/multiomics"
FILTERED_DIR <- file.path(WORKDIR, "filtered_data")   # written by 01_preprocess_eda.ipynb
BEFORE_DIR   <- file.path(WORKDIR, "before")
dir.create(BEFORE_DIR, showWarnings = FALSE, recursive = TRUE)

# Toggle: include the synthetic client_04_L03_L14 (L03 + L14 fold-in).
# Default FALSE -- only the three real cross-modality clients are prepared.
# MUST match the toggle in 01_preprocess_eda.ipynb (the upstream notebook
# decides whether client_04 rows are present in filtered_data/).
INCLUDE_CLIENT_04 <- FALSE

# Donor levels match the Quartet figshare design; D6 is the reference (intercept).
DONOR_LEVELS    <- c("D6", "D5", "F7", "M8")
DONOR_REFERENCE <- DONOR_LEVELS[1]
COVARIATES      <- DONOR_LEVELS[-1]

# Canonical client names (full federation; pruned below when the toggle is FALSE).
ALL_CLIENT_NAMES <- c("client_01_L01", "client_02_L02",
                     "client_03_L05_L04", "client_04_L03_L14")
OPTIONAL_CLIENTS <- c("client_04_L03_L14")
CLIENT_NAMES <- if (INCLUDE_CLIENT_04) {
  ALL_CLIENT_NAMES
} else {
  setdiff(ALL_CLIENT_NAMES, OPTIONAL_CLIENTS)
}
MODALITIES              <- c("Transcriptomics", "Proteomics", "Metabolomics")
# Validation constants follow the active client count (see 01_preprocess_eda.ipynb).
N_LIBS_PER_MODALITY     <- if (INCLUDE_CLIENT_04) 72L else 48L
N_BATCHES_PER_MODALITY  <- if (INCLUDE_CLIENT_04) 6L  else 4L
MIN_CLIENTS_PER_FEATURE <- 3L
MIN_SAMPLES_PER_FEATURE <- N_BATCHES_PER_MODALITY + length(COVARIATES) + 1L
PROTEOMICS_ZERO_Q3_PROB <- 0.75

cat("INCLUDE_CLIENT_04 =", INCLUDE_CLIENT_04, "-> active clients:",
    paste(CLIENT_NAMES, collapse = ", "), "\n")

INCLUDE_CLIENT_04 = FALSE -> active clients: client_01_L01, client_02_L02, client_03_L05_L04 


## Helper functions

`read_feature_matrix` reads a TSV written by `01_preprocess_eda.ipynb` (first
column = feature IDs → rownames, remaining columns = numeric samples).
`write_feature_matrix` writes the matrix in the same TSV format required by
the central and federated batch-correction apps. Proteomics client-wise
zero/Q3 filtering and the minimum-three-client feature rule are applied once
here and reused for both central and FedRBE. Metabolomics has no feature-based
filter: all metabolites from `filtered_data/Metabolomics_filtered.tsv` are
passed through unchanged, including original zeros, so central limma and
FedRBE see the full metabolite set.
`validate_filtered_design` verifies every structural invariant of the
pre-filtered data before writing `before/`.

In [3]:
# Read a TSV expression matrix saved by 01: first column becomes rownames.
read_feature_matrix <- function(path) {
  df <- readr::read_tsv(path, show_col_types = FALSE)
  m  <- as.matrix(df[, -1, drop = FALSE])
  rownames(m) <- df[[1]]
  m
}

# Write an expression matrix as TSV with a leading rowname column.
write_feature_matrix <- function(expr, path) {
  out <- as.data.frame(expr, check.names = FALSE) %>% rownames_to_column("rowname")
  write.table(out, file = path, sep = "\t", quote = TRUE, row.names = FALSE,
              col.names = TRUE, na = "NA")
}

# Proteomics encodes zero/missing-like values as the matrix minimum (-6.644 here).
# Filter per client by masking failed client-local feature values to NA, matching
# the soft filtering pattern used by other datasets.
filter_proteomics_zero_q3_by_client <- function(expr, metadata) {
  finite_values <- expr[is.finite(expr)]
  if (length(finite_values) == 0)
    stop("Proteomics: expression matrix contains no finite values.", call. = FALSE)

  zero_equivalent <- min(finite_values, na.rm = TRUE)
  cat("Proteomics zero-equivalent value:", zero_equivalent, "\n")

  masked_expr <- expr
  client_stats <- purrr::map_dfr(CLIENT_NAMES, function(client_name) {
    client_samples <- metadata %>%
      filter(client == client_name) %>%
      pull(file)
    if (length(client_samples) == 0)
      stop("Proteomics: no samples for ", client_name, call. = FALSE)

    client_expr <- expr[, client_samples, drop = FALSE]
    client_q3 <- apply(client_expr, 1, function(values) {
      values <- values[is.finite(values)]
      if (length(values) == 0) return(NA_real_)
      as.numeric(stats::quantile(values, probs = PROTEOMICS_ZERO_Q3_PROB,
                                 na.rm = TRUE, names = FALSE))
    })

    mask_features <- names(client_q3)[!is.finite(client_q3) | client_q3 <= zero_equivalent]
    if (length(mask_features) > 0)
      masked_expr[mask_features, client_samples] <<- NA_real_

    features_after <- nrow(client_expr) - length(mask_features)
    cat("Proteomics /", client_name, ": Q3-zero filter before", nrow(client_expr),
        "after", features_after, "masked", length(mask_features), "\n")
    tibble(
      client = client_name,
      features_before = nrow(client_expr),
      features_after = features_after,
      features_masked = length(mask_features)
    )
  })

  all_na_rows <- rowSums(!is.na(masked_expr)) == 0
  if (any(all_na_rows))
    cat("Proteomics : removed", sum(all_na_rows), "features that became all NA after client masking\n")
  masked_expr <- masked_expr[!all_na_rows, , drop = FALSE]
  cat("Proteomics : Q3-zero client-wise mask retained", nrow(masked_expr),
      "of", nrow(expr), "features\n")

  list(
    expr = masked_expr,
    zero_equivalent = zero_equivalent,
    features_before = nrow(expr),
    features_after = nrow(masked_expr),
    features_removed = sum(all_na_rows),
    features_masked_by_client = sum(client_stats$features_masked),
    client_stats = client_stats
  )
}

# Match the FeatureCloud app's local usability and global min-client rules.
# A feature is locally usable only after singleton observations within a batch
# are masked and the client still has enough observations for the global design.
filter_features_by_client_presence <- function(expr, metadata) {
  client_usable <- vapply(CLIENT_NAMES, function(client_name) {
    client_metadata <- metadata %>%
      filter(client == client_name)
    if (nrow(client_metadata) == 0)
      stop("No samples for ", client_name, call. = FALSE)

    client_expr <- expr[, client_metadata$file, drop = FALSE]
    for (batch_name in unique(client_metadata$batch)) {
      batch_samples <- client_metadata %>%
        filter(batch == batch_name) %>%
        pull(file)
      singleton_rows <- rowSums(!is.na(client_expr[, batch_samples, drop = FALSE])) == 1L
      client_expr[singleton_rows, batch_samples] <- NA_real_
    }

    rowSums(!is.na(client_expr)) >= MIN_SAMPLES_PER_FEATURE
  }, logical(nrow(expr)))

  client_counts <- rowSums(client_usable)
  keep <- client_counts >= MIN_CLIENTS_PER_FEATURE
  cat("Feature presence filter: retained", sum(keep), "of", nrow(expr),
      "features present in at least", MIN_CLIENTS_PER_FEATURE, "clients\n")

  list(
    expr = expr[keep, , drop = FALSE],
    features_before = nrow(expr),
    features_after = sum(keep),
    features_removed = sum(!keep),
    client_counts = client_counts
  )
}

design_for_client <- function(client_metadata) {
  out <- client_metadata
  for (covariate in COVARIATES) {
    out[[covariate]] <- as.integer(out$condition == covariate)
  }
  out %>%
    select(file, all_of(COVARIATES), batch, batch_code, condition, lab,
           platform, protocol, datatype, rep, date, pseudo_sample)
}

config_text <- function(position, reference_batch, smpc = TRUE) {
  reference <- if (isFALSE(reference_batch)) "false" else as.character(reference_batch)
  c(
    "flimmaBatchCorrection:",
    "  batch_col: batch",
    "  covariates:",
    paste0("  - ", COVARIATES),
    "  data_filename: intensities_log_UNION.tsv",
    "  design_filename: design.tsv",
    "  design_separator: \"\\t\"",
    "  expression_file_flag: true",
    "  index_col: rowname",
    "  min_samples: 0",
    "  normalizationMethod: null",
    paste0("  position: ", position),
    paste0("  reference_batch: ", reference),
    "  separator: \"\\t\"",
    paste0("  smpc: ", tolower(as.character(smpc)))
  )
}

write_fedrbe_client_inputs <- function(expr, metadata, omics, modality_dir) {
  purrr::map_dfr(seq_along(CLIENT_NAMES), function(client_idx) {
    client_name <- CLIENT_NAMES[[client_idx]]
    client_metadata <- metadata %>%
      filter(client == client_name) %>%
      arrange(lab, batch_code, condition, rep)
    if (nrow(client_metadata) == 0)
      stop(omics, ": no samples for ", client_name, call. = FALSE)

    client_expr <- expr[, client_metadata$file, drop = FALSE]
    all_na_rows <- rowSums(!is.na(client_expr)) == 0
    n_all_na_rows <- sum(all_na_rows)
    client_expr <- client_expr[!all_na_rows, , drop = FALSE]

    client_dir <- file.path(modality_dir, client_name)
    dir.create(client_dir, showWarnings = FALSE, recursive = TRUE)
    write_feature_matrix(client_expr, file.path(client_dir, "intensities_log_UNION.tsv"))
    write.table(design_for_client(client_metadata), file = file.path(client_dir, "design.tsv"),
                sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)

    batches <- sort(unique(client_metadata$batch))
    reference_batch <- if (client_idx == length(CLIENT_NAMES)) tail(batches, 1) else FALSE
    writeLines(config_text(client_idx - 1L, reference_batch, smpc = omics == "Proteomics"),
               file.path(client_dir, "config.yml"))

    tibble(
      modality = omics,
      client = client_name,
      position = client_idx - 1L,
      labs = paste(sort(unique(client_metadata$lab)), collapse = ","),
      n_labs = n_distinct(client_metadata$lab),
      n_batches = n_distinct(client_metadata$batch),
      batch_codes = paste(sort(unique(client_metadata$batch_code)), collapse = ","),
      batches = paste(batches, collapse = ","),
      n_samples = nrow(client_metadata),
      n_features = nrow(client_expr),
      all_na_feature_rows_dropped = n_all_na_rows,
      reference_batch = if (isFALSE(reference_batch)) "False" else reference_batch,
      path = file.path("before", omics, client_name)
    )
  })
}

# Full structural validation of the filtered design.
# Called on metadata read from filtered_data/ before writing before/.
validate_filtered_design <- function(metadata, label) {
  if (nrow(metadata) != N_LIBS_PER_MODALITY)
    stop(label, ": expected ", N_LIBS_PER_MODALITY, " samples, got ", nrow(metadata), call. = FALSE)
  if (n_distinct(metadata$batch) != N_BATCHES_PER_MODALITY)
    stop(label, ": expected ", N_BATCHES_PER_MODALITY, " batches.", call. = FALSE)
  if (!setequal(as.character(unique(metadata$condition)), DONOR_LEVELS))
    stop(label, ": unexpected condition levels: ", paste(unique(metadata$condition), collapse = ", "), call. = FALSE)
  donor_counts       <- metadata %>% count(condition)
  expected_per_donor <- N_LIBS_PER_MODALITY %/% length(DONOR_LEVELS)
  if (any(donor_counts$n != expected_per_donor))
    stop(label, ": expected ", expected_per_donor, " samples per donor.", call. = FALSE)
  batch_donor_counts <- metadata %>% count(batch, condition)
  bad            <- batch_donor_counts %>% filter(n != 3)
  expected_cells <- N_BATCHES_PER_MODALITY * length(DONOR_LEVELS)
  if (nrow(bad) > 0 || nrow(batch_donor_counts) != expected_cells)
    stop(label, ": expected exactly 3 replicates per donor in every batch.", call. = FALSE)
  client_counts <- metadata %>% count(client)
  if (!setequal(client_counts$client, CLIENT_NAMES))
    stop(label, ": expected exactly the active canonical clients (",
         paste(CLIENT_NAMES, collapse = ", "), ").", call. = FALSE)
  if (any(!client_counts$n %in% c(12L, 24L)))
    stop(label, ": each client must have 12 or 24 libraries.", call. = FALSE)
  invisible(TRUE)
}

## Write per-modality central and FedRBE inputs

For each modality, read the pre-filtered expression matrix and metadata from
`filtered_data/`, validate the design, sort samples deterministically by
`(client, batch, condition, rep)`, then write central and FedRBE inputs from
the same filtered matrices:
- `before/{Modality}/central_intensities_log_UNION.tsv` — full matrix
- `before/{Modality}/metadata.tsv` — aligned metadata
- `before/{Modality}/{client}/` — FedRBE client matrix, design, and config
- `before/fedrbe_client_groups.tsv` — client folder summary

In [4]:
# Load the combined metadata written by 01_preprocess_eda.ipynb (all modalities).
metadata_all <- readr::read_tsv(
  file.path(FILTERED_DIR, "metadata_filtered.tsv"),
  show_col_types = FALSE
)

fedrbe_client_summaries <- list()

prep_summaries <- purrr::map_dfr(MODALITIES, function(omics) {
  message("Preparing ", omics)

  # Read pre-filtered expression matrix saved by 01.
  expr     <- read_feature_matrix(file.path(FILTERED_DIR, paste0(omics, "_filtered.tsv")))
  metadata <- metadata_all %>%
    filter(datatype == omics) %>%
    mutate(condition = factor(condition, levels = DONOR_LEVELS))

  validate_filtered_design(metadata, omics)

  # Sort deterministically; restore condition to character for writing.
  metadata <- metadata %>%
    mutate(condition = as.character(condition)) %>%
    arrange(client, batch, condition, rep)
  expr <- expr[, metadata$file, drop = FALSE]

  zero_q3_features_before <- nrow(expr)
  zero_q3_features_after  <- nrow(expr)
  zero_q3_features_masked_by_client <- 0L
  zero_equivalent_value   <- NA_real_
  if (omics == "Proteomics") {
    zero_q3_filter <- filter_proteomics_zero_q3_by_client(expr, metadata)
    expr <- zero_q3_filter$expr
    zero_q3_features_before <- zero_q3_filter$features_before
    zero_q3_features_after  <- zero_q3_filter$features_after
    zero_q3_features_masked_by_client <- zero_q3_filter$features_masked_by_client
    zero_equivalent_value   <- zero_q3_filter$zero_equivalent
  }

  # Metabolomics: no feature-based filter (all metabolites pass through).
  client_presence_filter <- if (omics == "Metabolomics") {
    list(expr = expr, features_before = NA_integer_, features_after = NA_integer_,
         features_removed = NA_integer_)
  } else {
    filter_features_by_client_presence(expr, metadata)
  }
  expr <- client_presence_filter$expr

  # Variance check: zero-variance rows will be silently handled by limma but
  # flag them here so analysts are aware before running correction.
  variances <- apply(expr, 1, var, na.rm = TRUE)
  stats <- tibble(
    omics                    = omics,
    features                 = nrow(expr),
    zero_q3_features_before  = zero_q3_features_before,
    zero_q3_features_after   = zero_q3_features_after,
    zero_q3_features_removed = zero_q3_features_before - zero_q3_features_after,
    zero_q3_client_feature_masks = zero_q3_features_masked_by_client,
    zero_equivalent_value    = zero_equivalent_value,
    min_clients_per_feature  = MIN_CLIENTS_PER_FEATURE,
    client_presence_features_before = client_presence_filter$features_before,
    client_presence_features_after = client_presence_filter$features_after,
    client_presence_features_removed = client_presence_filter$features_removed,
    samples                  = ncol(expr),
    batches                  = n_distinct(metadata$batch),
    clients                  = n_distinct(metadata$client),
    donors                   = n_distinct(metadata$condition),
    missing_cells            = sum(is.na(expr)),
    rows_with_any_na         = sum(rowSums(is.na(expr)) > 0),
    zero_or_na_variance_rows = sum(!is.finite(variances) | variances == 0)
  )

  modality_dir <- file.path(BEFORE_DIR, omics)
  dir.create(modality_dir, showWarnings = FALSE, recursive = TRUE)
  write_feature_matrix(expr, file.path(modality_dir, "central_intensities_log_UNION.tsv"))
  write.table(metadata, file = file.path(modality_dir, "metadata.tsv"),
              sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)
  fedrbe_client_summaries[[omics]] <<- write_fedrbe_client_inputs(expr, metadata, omics, modality_dir)

  cat(omics, ": wrote", nrow(expr), "features x", ncol(expr),
      "samples to", normalizePath(modality_dir), "\n")
  stats
})

fedrbe_client_summary <- bind_rows(fedrbe_client_summaries)

prep_summaries

Preparing Transcriptomics



Feature presence filter: retained 26907 of 26907 features present in at least 3 clients
Transcriptomics : wrote 26907 features x 48 samples to /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/multiomics/before/Transcriptomics 


Preparing Proteomics



Proteomics zero-equivalent value: -6.644 
Proteomics / client_01_L01 : Q3-zero filter before 3489 after 2594 masked 895 
Proteomics / client_02_L02 : Q3-zero filter before 3489 after 3328 masked 161 
Proteomics / client_03_L05_L04 : Q3-zero filter before 3489 after 3447 masked 42 
Proteomics : removed 3 features that became all NA after client masking
Proteomics : Q3-zero client-wise mask retained 3486 of 3489 features
Feature presence filter: retained 2539 of 3486 features present in at least 3 clients
Proteomics : wrote 2539 features x 48 samples to /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/multiomics/before/Proteomics 


Preparing Metabolomics



Metabolomics : wrote 71 features x 48 samples to /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/multiomics/before/Metabolomics 


omics,features,zero_q3_features_before,zero_q3_features_after,zero_q3_features_removed,zero_q3_client_feature_masks,zero_equivalent_value,min_clients_per_feature,client_presence_features_before,client_presence_features_after,client_presence_features_removed,samples,batches,clients,donors,missing_cells,rows_with_any_na,zero_or_na_variance_rows
<chr>,<int>,<int>,<int>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
Transcriptomics,26907,26907,26907,0,0,NA,3,26907,26907,0,48,4,3,4,0,0,0
Proteomics,2539,3489,3486,3,1098,-6.644,3,3486,2539,947,48,4,3,4,0,0,0
Metabolomics,71,71,71,0,0,NA,3,NA,NA,NA,48,4,3,4,0,0,0


## Save preparation summary

The summary records matrix shapes and basic QC flags for all three modalities.
`datainfo.json` captures provenance for downstream correction scripts.

In [5]:
write.table(prep_summaries, file = file.path(BEFORE_DIR, "preparation_summary.tsv"),
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)
write.table(fedrbe_client_summary, file = file.path(BEFORE_DIR, "fedrbe_client_groups.tsv"),
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

datainfo <- list(
  source            = "Quartet full multiomics Figshare 10.6084/m9.figshare.22188349.v1",
  sample_set        = "four-client federation, 72 libraries per modality",
  correction_batch  = "batch",
  reference_donor   = DONOR_REFERENCE,
  preserved_biology = DONOR_LEVELS,
  client_names      = CLIENT_NAMES,
  modalities        = prep_summaries
)
writeLines(jsonlite::toJSON(datainfo, pretty = TRUE, auto_unbox = TRUE),
           file.path(BEFORE_DIR, "datainfo.json"))

cat("FedRBE client folders and groups written to", normalizePath(BEFORE_DIR), "\n")
cat("Prepared inputs written to", normalizePath(BEFORE_DIR), "\n")

FedRBE client folders and groups written to /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/multiomics/before 
Prepared inputs written to /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/multiomics/before 


## Session information

In [6]:
sessionInfo()

R version 4.5.2 (2025-10-31)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 24.04.4 LTS

Matrix products: default
BLAS/LAPACK: /home/yuliya-cosybio/miniforge3/envs/fedRBE/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Europe/Berlin
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] jsonlite_2.0.0  lubridate_1.9.5 forcats_1.0.1   stringr_1.6.0  
 [5] dplyr_1.2.1     purrr_1.2.1     readr_2.2.0     tidyr_1.3.2    
 [9] tibble_3.3.1    ggplot2_4.0.2   tidyverse_2.0.0

loaded via a namespace (and not a